# 第5日 ノートブック03：YOLO と PatchCore を比べる

# Day 5 Notebook 03: YOLO versus PatchCore

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 2 つの手法の出力の違いを整理できる | 1. Summarise how the two methods differ in output |
| 2. 同じ画像セットで両方の結果を並べられる | 2. Line up both results on the same image set |
| 3. 同じ基準で OK/NG を比べるときの注意を説明できる | 3. Explain the caution needed when comparing OK/NG decisions |
| 4. それぞれの長所・短所を製造の観点で説明できる | 4. Explain strengths and weaknesses from a manufacturing view |
| 5. PVC 検査にどちらを、どの順序で使うか提案できる | 5. Propose which to use and in what order for PVC |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 100 分 / 100 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日ノートブック03 と第5日ノートブック02 を実行し、`submissions` に結果が保存されていること。 / You ran Day 4 Notebook 03 and Day 5 Notebook 02, so the results exist in `submissions`. |
| 必要なライブラリ / Required libraries | pandas, numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | `aivision-anomalib`（YOLO の結果はファイルから読み込みます） |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第5日 / Day 5](../../COURSE_SYLLABUS.md#day5) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 比較表 | The comparison table |
| 並べた結果画像 | Side-by-side result images |
| PVC 検査への推奨（判断と理由） | A recommendation for PVC with reasons |

> YOLO and PatchCore live in two different environments, so this notebook compares the **saved results** instead of loading both models. That is a deliberate design choice, documented in `requirements/tested_versions.md`.

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第5日 / Day 5 |
| この日の順番 / Position today | 3 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 100 分 / 100 minutes |
| 前提 / Prerequisites | `Day04_03` と `Day05_02` の保存済み結果 |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第5日 / Day 5](../../COURSE_SYLLABUS.md#day5) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


### 予想して、実行して、説明する / Predict, run, inspect, explain

較正は判定条件を決める工程です。試験は固定後に別データで確認する工程です。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。較正用スライダーだけを変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

Calibration selects the rule; test checks it on different data after freezing. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only the calibration slider, predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(13, 4.5)):
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

---

## Part 1　出力の違いをおさらいする / Recap: the outputs differ

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
show([Image.open(ASSETS / "diagrams" / "diagram_yolo_vs_patchcore.png")],
     ["two different answers to two different questions"], figsize=(11, 5.5))

#### 同じにできないものは、同じに扱わない
2 つの手法は、**返すものが違います**。

| | YOLO | PatchCore |
|---|---|---|
| 主な出力 | 四角 + クラス + confidence | 異常スコア + ヒートマップ |
| 「何の欠陥か」 | **分かる** | 分からない |
| 「どこか」 | 四角の位置 | ヒートマップの明るい場所 |
| 学習に要るもの | 欠陥のラベル | 正常と確認できた画像 |

したがって、**mAP と AUROC を直接比べてはいけません**。
測っているものが違うからです。

比べてよいのは、**同じ正解ラベルに対して、それぞれの手法が画像単位で出した OK/NG の判定** です。
そのためには、**それぞれの手法に判定規則（しきい値）を宣言し、両方を同じ正解と突き合わせます**。
2 つの手法どうしが一致しているかだけを見ると、**両方が同じように間違っている場合** を見逃します。

#### Do not pretend different things are the same
The two methods **return different things**:

| | YOLO | PatchCore |
|---|---|---|
| main output | box + class + confidence | anomaly score + heatmap |
| "which defect" | **yes** | no |
| "where" | box location | bright regions of the heatmap |
| fitting needs | defect labels | images confirmed as normal |

Therefore **mAP and AUROC must not be compared directly** — they measure different things.

What *can* be compared is **each method's image-level OK/NG decision against the same ground
truth**, once **each method has a declared decision rule (threshold)**. Comparing only whether the
two methods agree would hide the case where **both are wrong in the same way**.


---

## Part 2　保存された結果を読み込む / Loading the saved results

### 評価の順番 / the order of evaluation

![](../../assets/diagrams/teach_evaluation_sequence.png)

図：分割 → 学習 → 較正 → 固定 → 評価 → 報告 の順番（模式図）。較正から報告へ直接進む近道はありません。

Figure: the order split → fit → calibrate → freeze → evaluate → report (schematic). There is no shortcut from calibration straight to the report.

**気づいてほしいこと**

④ で固定した後にしきい値を動かすと、⑤ の数字は「試験の成績」ではなくなります。同じ設定での再評価はできますが、試験結果を見て設定を変えたら、新しい未使用データが必要です。

**What to notice**

Moving the threshold after the freeze in ④ turns the numbers in ⑤ into something other than a test result. You may repeat an unchanged evaluation; if you change settings after seeing the test results, you need a new untouched holdout.

**考えてみてください / Try to answer**

- 固定するのは、モデル・前処理・しきい値のどれですか。
- What gets frozen: the model, the preprocessing, the threshold — or all three?


#### 3 つのファイルを読む

| ファイル | 作った場所 | 何が入っているか |
|---|---|---|
| `submissions/day05/split_manifest.json` | 第5日ノートブック02 | **分割の記録**。どの画像が学習・較正・試験か |
| `submissions/day04/yolo_predictions_on_shared_set.json` | 第4日ノートブック03 | 全画像の YOLO 検出（信頼度つき）|
| `submissions/day05/patchcore_predictions.json` | 第5日ノートブック02 | 全画像の PatchCore スコア |

**分割の記録が無い結果は使いません。** どのデータでしきい値を決めたか分からない数字は、比較の材料になりません。古い形式のファイルは、理由を表示して止めます。

#### Read the three files

| File | Produced by | Contents |
|---|---|---|
| `submissions/day05/split_manifest.json` | Day 5 Notebook 02 | **the split record**: which image belongs to train, calibration or test |
| `submissions/day04/yolo_predictions_on_shared_set.json` | Day 4 Notebook 03 | YOLO detections with confidences, for every image |
| `submissions/day05/patchcore_predictions.json` | Day 5 Notebook 02 | PatchCore scores for every image |

**A result without its split record is not used.** A number whose calibration data is unknown cannot support a comparison. Outdated files stop the notebook with an explanation.

In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import json
from pathlib import Path

manifest_path = COURSE_ROOT / "submissions" / "day05" / "split_manifest.json"
yolo_path = COURSE_ROOT / "submissions" / "day04" / "yolo_predictions_on_shared_set.json"
patchcore_path = COURSE_ROOT / "submissions" / "day05" / "patchcore_predictions.json"

REQUIRED_SCHEMA = 2
missing = [p for p in (manifest_path, yolo_path, patchcore_path) if not p.is_file()]
if missing:
    for path in missing:
        print("見つかりません / missing:", path)
    print()
    print("先に実行してください / run first:")
    print("  notebooks/Day04_YOLO_Training/Day04_03_YOLO_Evaluation.ipynb")
    print("  notebooks/Day05_PatchCore/Day05_02_PatchCore_Training.ipynb")
    raise FileNotFoundError(
        "比較に必要なファイルがそろっていません / the files needed for the comparison are missing"
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
yolo_data = json.loads(yolo_path.read_text(encoding="utf-8"))
patchcore_data = json.loads(patchcore_path.read_text(encoding="utf-8"))

stale = []
for name, payload in (("split_manifest.json", manifest),
                      ("yolo_predictions_on_shared_set.json", yolo_data),
                      ("patchcore_predictions.json", patchcore_data)):
    version = payload.get("schema_version")
    if version != REQUIRED_SCHEMA:
        stale.append(f"{name}: schema_version={version!r}（必要 / required: {REQUIRED_SCHEMA}）")
if stale:
    print("古い形式の結果ファイルです / these result files are from an older method:")
    for line in stale:
        print("  -", line)
    print()
    print("古い比較結果を上書きして「新しい方法で出した数字」として扱うことはできません。")
    print("第4日ノートブック03 と第5日ノートブック02 を実行し直してください。")
    print("Old comparisons must not be overwritten and passed off as new results.")
    print("Re-run Day 4 Notebook 03 and Day 5 Notebook 02.")
    raise ValueError("stale result files / 古い結果ファイル")

print("分割の件数 / partition sizes:", manifest["counts"])
print("分割の重なり / overlap check  :", manifest["overlap_check"])
print("スコアの決まり / score convention:", manifest["score_convention"][:70], "...")

#### フレームにする

`image` 列はフォルダをまたいで同じ書き方にそろえます（`画像の種類/ファイル名.png`）。そろえないと、同じ画像を 2 つの手法で結び付けられません。

#### Turn them into frames

The `image` column is normalised to `type/file.png` for both methods. Without that, the same image cannot be matched across the two methods.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import pandas as pd

yolo_frame = pd.DataFrame(yolo_data["images"])
patchcore_frame = pd.DataFrame(patchcore_data["images"])

yolo_frame["image"] = yolo_frame["image"].astype(str)
patchcore_frame["image"] = patchcore_frame["image"].astype(str).str.replace("^test/", "", regex=True)
yolo_frame["image"] = yolo_frame["image"].str.replace("^test/", "", regex=True)

partition_of = {}
for partition, files in manifest["partitions"].items():
    for name in files:
        partition_of[name.replace("test/", "", 1)] = partition

yolo_frame["truly_abnormal"] = yolo_frame["true_label"].astype(str).eq("bad")
yolo_frame["partition"] = yolo_frame["image"].map(partition_of)
patchcore_frame["partition"] = patchcore_frame["image"].map(partition_of)

print("YOLO の行数 / YOLO rows          :", len(yolo_frame))
print("PatchCore の行数 / PatchCore rows:", len(patchcore_frame))
print("分割が決まらない行 / rows without a partition:",
      int(yolo_frame["partition"].isna().sum()), int(patchcore_frame["partition"].isna().sum()))
print()
print("推論した最小 confidence / minimum inference confidence:",
      yolo_data.get("min_inference_conf"))
print("YOLO の判定規則 / YOLO decision rule:", yolo_data.get("decision_rule"))

## Part 3　同じ画像で並べる / Lining them up on the same images

#### 同じ画像で、両方の結果がそろっている画像だけを使う

片方の手法しか結果が無い画像は、**比べられません**。黙って捨てると「都合のよい画像だけの比較」になります。だから、まず **何枚そろっていて、何枚足りないか** を数えて表示します。

#### Use only images where both methods produced a result

An image with only one method's result **cannot be compared**. Dropping it silently would leave a comparison of convenient images only. So count and show **how many images match and how many are missing** before comparing anything.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
REQUIRED_COLUMNS = ["partition", "image", "truly_abnormal"]

problems = []
for name, frame in (("YOLO", yolo_frame), ("PatchCore", patchcore_frame)):
    absent = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
    if absent:
        problems.append(f"{name}: 列がありません / missing columns {absent}")
if problems:
    print("結果ファイルの形が想定と違います / unexpected result-file shape:")
    for line in problems:
        print("  -", line)
    print("第4日ノートブック03 と第5日ノートブック02 を実行し直してください。")
    raise ValueError("unexpected result-file shape")

yolo_images = set(yolo_frame["image"])
patchcore_images = set(patchcore_frame["image"])
shared_images = yolo_images & patchcore_images

coverage = pd.DataFrame({
    "partition": sorted(set(yolo_frame["partition"].dropna())),
})
coverage["in manifest"] = [
    len(manifest["partitions"][p]) for p in coverage["partition"]
]
coverage["YOLO rows"] = [
    int((yolo_frame["partition"] == p).sum()) for p in coverage["partition"]
]
coverage["PatchCore rows"] = [
    int((patchcore_frame["partition"] == p).sum()) for p in coverage["partition"]
]
print(coverage.to_string(index=False))
print()

missing_yolo = sorted(patchcore_images - yolo_images)
missing_patchcore = sorted(yolo_images - patchcore_images)
print("両方そろった画像 / images with both results:", len(shared_images))
print("YOLO の結果が無い画像 / no YOLO result        :", len(missing_yolo))
print("PatchCore の結果が無い画像 / no PatchCore result:", len(missing_patchcore))

if missing_yolo:
    print()
    print("YOLO の結果が無い画像（比較から外れます）/ images excluded for lack of YOLO:")
    for name in missing_yolo[:5]:
        print("  ", name)
if missing_patchcore:
    print()
    print("PatchCore の結果が無い画像（比較から外れます）/ excluded for lack of PatchCore:")
    for name in missing_patchcore[:5]:
        print("  ", name)

if not shared_images:
    raise RuntimeError(
        "共通の画像が 1 枚もありません / there are no images with both results. "
        "第4日ノートブック03 と第5日ノートブック02 の出力先を確認してください。"
    )

#### 分割ごとに分ける

| 分割 | 何に使うか | いつ見てよいか |
|---|---|---|
| `patchcore_train` | PatchCore の学習（正常のみ）| 学習のときだけ |
| `calibration` | **しきい値と判定規則を決める** | 何度でも見てよい |
| `test` | 最後に 1 回だけ成績を出す | **モデル・前処理・しきい値を固定してから** |

見てよいかどうかは **「情報をどう使ったか」** で決まります。試験用データを見てからしきい値や
前処理を変えると、そのデータは「まだ見ていないデータ」ではなくなります。固定した設定で
同じ評価をもう一度実行するのは構いません（再現性の確認です）。

この研修の参考図は、**保存した 1 回の実行から作った公開済みの結果** です。参考図で試験用の例を
見てから設定を変えた場合、自分の手元の試験用データはもう未使用とは言えません。
新しい最終成績を出すには、**新しい未使用データ** が必要です。

#### Split the images into their roles

| Partition | What it is for | When may you look? |
|---|---|---|
| `patchcore_train` | PatchCore fitting (normal only) | only while fitting |
| `calibration` | **choose the thresholds and the decision rule** | as often as you like |
| `test` | the final score, once | **after the model, the preprocessing and the thresholds are frozen** |

What matters is **how the information is used**. If you change a threshold or the preprocessing
after studying the test data, that data is no longer unseen. Repeating the same evaluation with the
frozen configuration is fine — that is a reproducibility check.

The reference figures in this course are **published results of one recorded run**. If you change
settings after studying the test examples in those figures, your own test partition is no longer
untouched. **A new final result needs a fresh holdout.**


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
def as_frame(frame, columns):
    subset = frame[frame["image"].isin(shared_images)].copy()
    subset = subset.drop_duplicates("image").set_index("image")
    return subset[columns]


yolo_side = as_frame(yolo_frame, ["partition", "truly_abnormal", "yolo_max_conf"])
patchcore_side = as_frame(patchcore_frame, ["partition", "truly_abnormal", "score"])

combined = yolo_side.join(patchcore_side, rsuffix="_patchcore", how="inner")
combined = combined.rename(columns={"truly_abnormal": "truth_ng",
                                    "truly_abnormal_patchcore": "truth_ng_patchcore"})
mismatch = combined[combined["truth_ng"] != combined["truth_ng_patchcore"]]
if len(mismatch):
    print("正解ラベルが 2 つのファイルで食い違っています / ground truth disagrees between the two files:")
    print(mismatch[["truth_ng", "truth_ng_patchcore"]].head().to_string())
    raise ValueError("ground-truth mismatch / 正解ラベルの食い違い")

combined["truth_ng"] = combined["truth_ng"].astype(bool)

calibration_frame = combined[combined["partition"] == "calibration"].copy()
test_frame = combined[combined["partition"] == "test"].copy()

print("較正用 / calibration :", len(calibration_frame),
      "（正常 / normal:", int((~calibration_frame["truth_ng"]).sum()),
      ", 異常 / abnormal:", int(calibration_frame["truth_ng"].sum()), "）")
print("試験用 / test        :", len(test_frame),
      "（正常 / normal:", int((~test_frame["truth_ng"]).sum()),
      ", 異常 / abnormal:", int(test_frame["truth_ng"].sum()), "）")

# 学習画像が試験用に混ざっていないことを、ファイル名で再確認します。
# 同じファイルが 2 つのフォルダに入っていても、この確認で見つかります。
# Re-check by file name: this also catches the same file stored in two folders.
train_names = {Path(name).name for name in manifest["partitions"]["patchcore_train"]}
test_names = {Path(name).name for name in manifest["partitions"]["test"]}
calibration_names = {Path(name).name for name in manifest["partitions"]["calibration"]}
leak = sorted((train_names & test_names) | (train_names & calibration_names))
assert not leak, f"学習画像が評価用に混ざっています / training images inside an evaluation partition: {leak[:3]}"
print()
print("学習画像と評価用のファイル名の重なり / training file names inside an evaluation partition:",
      len(leak))

## Part 4　同じ基準で OK/NG を出す / Making a common OK/NG decision

#### しきい値は「較正用」で決める

前の版では、正常スコアの最大と異常スコアの最小からしきい値を決め、**同じデータで成績を出して** いました。これは「答えを見てから採点する」のと同じで、数字が良く出ます。やめます。

いまは 2 段階です。

1. **較正用（calibration）** で 2 つのしきい値を決める。目的は 1 つ。
   **正常品の誤報率（false reject rate）を 10% 以下に保ったまま、再現率（recall）を最大にする。**
   この 10% は **この研修の目標値** です。較正用の正常品は 6 枚しかないので、10% は
   「誤報 0 枚」と同じ意味になります。**生産での誤報率を保証する数字ではありません。**
2. 決めた値を **固定** し、**試験用（test）** で 1 回だけ成績を出す。

| 手法 | 判定規則 | 決める値 |
|---|---|---|
| YOLO | `confidence ≥ C` の検出が 1 つ以上あれば NG | `C` |
| PatchCore | `score ≥ T` なら NG | `T` |

2 つのしきい値は **そのままでは比べられません**。測っている量が違うからです
（YOLO は confidence、PatchCore は較正で正規化したスコアです）。たまたま同じ数値になることも
ありますが、それは「同じ意味」ではありません。比べるのは数値ではなく、
**同じ正解に対する画像単位の OK/NG の判定** です。

#### Choose the thresholds on the calibration partition

The earlier version derived a threshold from the maximum normal score and the minimum abnormal score **and then scored the same data** — the equivalent of marking your own exam. That is gone.

Now there are two steps:

1. On **calibration**, choose both thresholds with one shared objective:
   **maximise recall subject to a false-reject rate of at most 10%.** The 10% figure is a
   **classroom objective**; with only six normal calibration images it means "zero observed false
   rejects". It is **not** a production-rate guarantee.
2. **Freeze** them and score **test** exactly once.

| Method | Decision rule | Value chosen |
|---|---|---|
| YOLO | NG if at least one detection with `confidence >= C` | `C` |
| PatchCore | NG if `score >= T` | `T` |

The two thresholds are **not directly comparable**: they measure different quantities (a detection
confidence and a score normalised on calibration). They could coincidentally hold the same number,
which would not make them the same thing. What is compared is **the image-level OK/NG decision
against the same ground truth**.


### 較正用データの分布 / the distribution on calibration data

![](../../assets/reference_outputs/ref_score_distribution.png)

図：較正用（validation）データだけのスコア分布と、そこから決めたしきい値 0.50（保存値の 0〜1 スケール）。試験用データは含みません。

Figure: the score distribution on the calibration (validation) partition only, with the threshold 0.50 chosen there, on the saved 0–1 scale. The test partition is not included.

**気づいてほしいこと**

スコアは anomalib 2.7.0 の後処理で `clip((生の値 − しきい値) ÷ (較正の最大 − 最小) + 0.5, 0, 1)` に変換されており、**学習したしきい値が 0.5** になります。較正は正常 6 枚・異常 20 枚です。6 枚で誤報 0 件は「本番でも 0 件」の保証ではありません。

**What to notice**

anomalib 2.7.0's post-processing maps the score with `clip((raw − threshold) / (cal_max − cal_min) + 0.5, 0, 1)`, so the **learned threshold becomes 0.5**. Calibration holds 6 normal and 20 abnormal images; zero false alarms in six does not guarantee zero in production.

**考えてみてください / Try to answer**

- しきい値を 0.60 にすると、この較正データでは見逃しが何枚増えますか。
- At a threshold of 0.60, how many misses appear on this calibration data?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
MAX_FALSE_REJECT_RATE = 0.10   # 正常品の誤報率の上限 / acceptable false-reject rate

# YOLO は 0.01 で推論してあるので、0.01 未満のしきい値は指定できません。
# YOLO was inferred at 0.01, so no threshold below 0.01 exists.
YOLO_MIN_CONF = float(yolo_data["min_inference_conf"])


def counts(predicted_ng, truth_ng):
    predicted_ng = pd.Series(predicted_ng).astype(bool)
    truth_ng = pd.Series(truth_ng).astype(bool)
    return {
        "TP": int((predicted_ng & truth_ng).sum()),
        "FP": int((predicted_ng & ~truth_ng).sum()),
        "FN": int((~predicted_ng & truth_ng).sum()),
        "TN": int((~predicted_ng & ~truth_ng).sum()),
    }


def choose_threshold(candidates, scores, truth, rule):
    """共通の目的関数で 1 つのしきい値を選ぶ / pick one threshold with the shared objective."""
    best = None
    for threshold in candidates:
        predicted = rule(scores, threshold)
        table = counts(predicted, truth)
        normals = table["FP"] + table["TN"]
        abnormal = table["TP"] + table["FN"]
        false_reject_rate = table["FP"] / normals if normals else float("nan")
        recall = table["TP"] / abnormal if abnormal else float("nan")
        if not normals or false_reject_rate > MAX_FALSE_REJECT_RATE:
            continue
        if not abnormal:
            continue
        key = (recall, threshold)
        if best is None or key > best["key"]:
            best = {"key": key, "threshold": float(threshold), "table": table,
                    "false_reject_rate": false_reject_rate, "recall": recall}
    return best


yolo_candidates = [YOLO_MIN_CONF + i * 0.01 for i in range(int((0.99 - YOLO_MIN_CONF) / 0.01) + 1)]
patchcore_candidates = [i / 100 for i in range(0, 101)]

yolo_best = choose_threshold(
    yolo_candidates,
    calibration_frame["yolo_max_conf"],
    calibration_frame["truth_ng"],
    lambda scores, threshold: scores >= threshold,
)
patchcore_best = choose_threshold(
    patchcore_candidates,
    calibration_frame["score"],
    calibration_frame["truth_ng"],
    lambda scores, threshold: scores >= threshold,
)

assert yolo_best is not None, "較正用データで目的を満たす YOLO しきい値がありません"
assert patchcore_best is not None, "較正用データで目的を満たす PatchCore しきい値がありません"

YOLO_THRESHOLD = yolo_best["threshold"]
PATCHCORE_THRESHOLD = patchcore_best["threshold"]

calibration_summary = pd.DataFrame([
    {"method": "YOLO", "threshold": round(YOLO_THRESHOLD, 3),
     "recall": round(yolo_best["recall"], 3),
     "false_reject_rate": round(yolo_best["false_reject_rate"], 3),
     **yolo_best["table"]},
    {"method": "PatchCore", "threshold": round(PATCHCORE_THRESHOLD, 3),
     "recall": round(patchcore_best["recall"], 3),
     "false_reject_rate": round(patchcore_best["false_reject_rate"], 3),
     **patchcore_best["table"]},
])
n_calibration_normal = int((~calibration_frame["truth_ng"]).sum())
max_allowed_fp = int(MAX_FALSE_REJECT_RATE * n_calibration_normal)
print(f"較正用の正常品は {n_calibration_normal} 枚なので、誤報率 "
      f"{MAX_FALSE_REJECT_RATE:.0%} は「誤報 {max_allowed_fp} 枚まで」という意味です。")
print(f"{n_calibration_normal} calibration normals -> the {MAX_FALSE_REJECT_RATE:.0%} cap allows "
      f"at most {max_allowed_fp} false alarm(s).")
print()
print("目的 / objective: recall を最大に、ただし正常品の誤報率 ≤ "
      f"{MAX_FALSE_REJECT_RATE:.0%} / maximise recall subject to false rejects ≤ "
      f"{MAX_FALSE_REJECT_RATE:.0%}")
print("（較正用データでの結果です / measured on the calibration partition）")
print()
print(calibration_summary.to_string(index=False))
print()
print("この 2 つの値を固定します / freezing these two values:")
print("  YOLO confidence threshold      :", round(YOLO_THRESHOLD, 3))
print("  PatchCore anomaly threshold    :", round(PATCHCORE_THRESHOLD, 3))

### コードの読み方 / Reading the code

![](../../assets/svg/teach_frozen_threshold_pipeline.svg)

図：試験から調整へ戻る矢印はありません。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: There is no feedback from test into tuning. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

試験から調整へ戻る矢印はありません。 入力と変換後の値を指でたどってください。

**What to notice**

There is no feedback from test into tuning. Trace the input and transformed values.

**考えてみてください / Try to answer**

試験結果を見て設定を変えたら、最終評価には何が必要ですか。

After tuning on test results, what is needed for a final evaluation?


#### 固定したしきい値で、試験用を 1 回だけ評価する

しきい値は較正用データで決めて、**ここで固定** しました。これから先、試験用の数字を見て
しきい値を動かすと、その表は「答えを見てから採点した表」になります。
動かしたくなったら、次の節のスライダーを **較正用データ** で使ってください。
新しい設定で最終成績を出したいときは、**別の未使用データ** を用意してください。

#### Score the test partition once, with the frozen thresholds

The thresholds were chosen on calibration and are **frozen here**. Tuning them after seeing these
numbers turns this table into marking your own exam. If you want to move one, use the slider in the
next section — on the **calibration** data. If you want a final result for a revised configuration,
**prepare a fresh holdout**.

**重み・前処理・正常メモリ・後処理も固定**します。試験を見てこれらを変えてはいけません。
同梱図の 0.06 / 0.50 は保存済み参考実行の例で、今回の実行の期待値ではありません。

**Freeze weights, preprocessing, normal memory and postprocessing too.** Do not change them
after inspecting test results. Supplied 0.06 / 0.50 values are saved reference-run examples,
not expected results for your live run.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json
from datetime import datetime, timezone

test_predictions = {
    "YOLO": test_frame["yolo_max_conf"] >= YOLO_THRESHOLD,
    "PatchCore": test_frame["score"] >= PATCHCORE_THRESHOLD,
}
truth = test_frame["truth_ng"]

rows = []
for method, predicted in test_predictions.items():
    table = counts(predicted, truth)
    tpr = table["TP"] / (table["TP"] + table["FN"]) if (table["TP"] + table["FN"]) else float("nan")
    fpr = table["FP"] / (table["FP"] + table["TN"]) if (table["FP"] + table["TN"]) else float("nan")
    precision = table["TP"] / (table["TP"] + table["FP"]) if (table["TP"] + table["FP"]) else float("nan")
    rows.append({
        "method": method,
        "threshold": round(YOLO_THRESHOLD if method == "YOLO" else PATCHCORE_THRESHOLD, 3),
        **table,
        "n_images": table["TP"] + table["FP"] + table["FN"] + table["TN"],
        "n_abnormal": table["TP"] + table["FN"],
        "n_normal": table["FP"] + table["TN"],
        "recall": round(tpr, 3) if tpr == tpr else float("nan"),
        "false_reject_rate": round(fpr, 3) if fpr == fpr else float("nan"),
        "precision": round(precision, 3) if precision == precision else float("nan"),
    })

comparison = pd.DataFrame(rows)
print("試験用データでの結果（しきい値は固定済み）/ results on the untouched test partition")
print("分母 / denominators: 異常 n=", int((truth).sum()), ", 正常 n=", int((~truth).sum()))
print()
print(comparison.to_string(index=False))
print()
if comparison["precision"].isna().any() or comparison["recall"].isna().any():
    print("注意 / caution: nan は「0 件で計算できない」という意味です。0 として扱わないでください。")
    print("nan means the denominator was empty. It does not mean zero.")

# 欠陥の種類ごとの内訳（件数が少ないので、参考として読むこと）
per_type = []
defect_types = test_frame.index.str.split("/").str[0]
for method, predicted in test_predictions.items():
    for name in sorted(set(defect_types)):
        mask = defect_types == name
        table = counts(predicted[mask], truth[mask])
        per_type.append({"method": method, "defect_type": name, **table})
per_type_frame = pd.DataFrame(per_type)
print()
print("欠陥の種類ごと（試験用）/ per defect type on the test partition")
print(per_type_frame.to_string(index=False))

# 古い比較結果は「この方法で出した結果」ではないので、上書きせず別名で保存します。
# The old comparison was produced by a different method: never overwrite it.
results = {
    "schema_version": 2,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "supersedes": {
        "file": "yolo_vs_patchcore.csv",
        "status": "superseded",
        "reason": (
            "The earlier comparison chose the PatchCore threshold from the maximum normal and "
            "minimum abnormal score of the same merged dataset it then evaluated. It is kept as "
            "historical evidence and must not be read as a result of the current method."
        ),
    },
    "split_manifest": str(manifest_path.relative_to(COURSE_ROOT)),
    "objective": {
        "function": "maximise recall subject to false_reject_rate <= MAX_FALSE_REJECT_RATE",
        "max_false_reject_rate": MAX_FALSE_REJECT_RATE,
        "selected_on": "calibration partition only",
    },
    "calibration": calibration_summary.to_dict(orient="records"),
    "test": comparison.to_dict(orient="records"),
    "per_defect_type": per_type_frame.to_dict(orient="records"),
    "score_convention": manifest["score_convention"],
    "model_identity": manifest["model"],
    "yolo_weights": yolo_data.get("weights"),
    "yolo_min_inference_conf": YOLO_MIN_CONF,
    "decisions": {
        "YOLO": "NG if at least one detection with conf >= threshold",
        "PatchCore": "NG if image anomaly score >= threshold",
    },
    "coverage": {
        "images_with_both_results": len(shared_images),
        "excluded_no_yolo": missing_yolo,
        "excluded_no_patchcore": missing_patchcore,
    },
}

results_path = COURSE_ROOT / "submissions" / "day05" / "comparison_test_results.json"
results_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
comparison.to_csv(COURSE_ROOT / "submissions" / "day05" / "comparison_test_results.csv",
                  index=False, encoding="utf-8-sig")
print()
print("保存 / saved:", results_path)

#### しきい値を動かしてよいのは、較正用データだけ

下のスライダーは **較正用データ** のスコア分布だけを使います。試験用のスコアは読み込んでいません。
**較正用の上で何度試しても、上の試験用の表は変わりません。**

ただし、**すでに試験用の結果を見てしまったあとでは**、較正用のスライダーを動かして選び直しても、
その判断は「試験用の結果を知った人」が下したものになります。**新しい最終成績には、新しい
未使用データが必要**です。

誤報（FP）と見逃し（FN）は **いつも同じ方向に動くわけではありません**。
しきい値を上げると見逃しは増え、誤報は減ります。**同時に 0 にできる場合もあります** — 正常品の
スコアの最大値より、異常品のスコアの最小値が大きい（分布に隙間がある）ときです。
この研修で保存した実行では、較正用データにその隙間があったので、PatchCore は 0.50 で
**較正用の 26 枚（正常 6・異常 20）で** 見逃し 0・誤報 0 になりました。
**これは 1 回の記録された実行の値** で、どの環境でも同じにはなりません。
また、**しきい値を動かしても件数が変わらない区間** もあります（その間に入る画像が無いため）。
隙間が無いデータでは、どちらかを犠牲にするしかありません。

#### You may move thresholds on the calibration data only

The slider below uses **calibration** scores only; the test scores are not loaded into it.
**Trying as many values as you like on calibration does not change the test table above.**

Once you have seen the test results, however, a choice made with the calibration slider is still a
choice made by someone who knows the test outcome. **A new final result needs a fresh holdout.**

False alarms (FP) and misses (FN) do **not** always move in the same direction. Raising the
threshold adds misses and removes false alarms, and **both can reach zero at once** when the
distributions have a gap — the lowest abnormal score above the highest normal score. In the run
recorded for this course the calibration data had that gap, so PatchCore reached zero misses and
zero false alarms at 0.50 **on the 26 calibration images (6 normal, 20 abnormal)**. **That is one
recorded run**, not a result every machine reproduces. **Some threshold intervals change no counts at
all**, because no image falls between them. Without a gap you must sacrifice one side.


### 較正で決めて、固定する / choose on calibration, then freeze

![](../../assets/diagrams/teach_threshold_freeze.png)

図：較正用データでしきい値を決めて固定する流れ（模式図）。この研修の固定値は YOLO 0.06、PatchCore 0.50 です。

Figure: choosing a threshold on calibration data, then freezing it (schematic). This course froze YOLO at 0.06 and PatchCore at 0.50.

**気づいてほしいこと**

固定した後に試験用データを見て動かすと、その数字はもう「未使用データでの成績」ではありません。較正用データは正常 6 枚・異常 20 枚なので、割合は 1/6 刻みでしか動きません。

**What to notice**

If you move the threshold after seeing the test data, those numbers are no longer a result on untouched data. Calibration holds 6 normal images, so a false-reject rate can only move in steps of 1/6.

**考えてみてください / Try to answer**

- 較正用の正常が 6 枚のとき、誤報率 10% 以下にするには誤報が何件以下である必要がありますか。
- With six normal calibration images, how many false alarms are allowed for a false-reject rate of at most 10 %?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# ウィジェットが無い環境のためのパラメータセル / parameter fallback: no widgets needed
import matplotlib.pyplot as plt

patchcore_threshold_to_try = 0.50   # ← この値を変える / change this value

predicted = calibration_frame["score"] >= patchcore_threshold_to_try
table = counts(predicted, calibration_frame["truth_ng"])
normals = table["FP"] + table["TN"]
abnormal = table["TP"] + table["FN"]
print(f"PatchCore しきい値 / threshold = {patchcore_threshold_to_try:.2f}")
print(f"較正用の誤報 FP = {table['FP']} / {normals}  見逃し FN = {table['FN']} / {abnormal}")
print("誤報率 / false-reject rate =",
      round(table["FP"] / normals, 3) if normals else "n/a")
print("再現率 / recall            =",
      round(table["TP"] / abnormal, 3) if abnormal else "n/a")

fig, ax = plt.subplots(figsize=(9.5, 3.6))
bins = [i / 50 for i in range(51)]
ax.hist(calibration_frame.loc[~calibration_frame["truth_ng"], "score"], bins=bins,
        color="#8fb6ff", edgecolor="#1f6feb", label="正常 / normal (calibration)")
ax.hist(calibration_frame.loc[calibration_frame["truth_ng"], "score"], bins=bins,
        color="#ffd580", edgecolor="#9a6700", label="異常 / abnormal (calibration)")
ax.axvline(patchcore_threshold_to_try, color="#a40e26", linewidth=2)
ax.set_xlabel("PatchCore 異常スコア / anomaly score（較正用データの正規化済み / normalised on calibration）")
ax.set_ylabel("枚数 / images")
ax.legend(fontsize=8)
ax.set_title(f"FP={table['FP']}  FN={table['FN']}（較正用 / calibration）", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# スライダー版（較正用データのみ）。ウィジェットが無ければ下のパラメータセルを使ってください。
if not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    output = widgets.Output()

    def draw(threshold):
        predicted = calibration_frame["score"] >= threshold
        table = counts(predicted, calibration_frame["truth_ng"])
        normals = table["FP"] + table["TN"]
        abnormal = table["TP"] + table["FN"]
        fig, ax = plt.subplots(figsize=(9.5, 3.6))
        bins = [i / 50 for i in range(51)]
        ax.hist(calibration_frame.loc[~calibration_frame["truth_ng"], "score"], bins=bins,
                color="#8fb6ff", edgecolor="#1f6feb", label="正常 / normal (calibration)")
        ax.hist(calibration_frame.loc[calibration_frame["truth_ng"], "score"], bins=bins,
                color="#ffd580", edgecolor="#9a6700", label="異常 / abnormal (calibration)")
        ax.axvline(threshold, color="#a40e26", linewidth=2)
        ax.set_xlabel("PatchCore 異常スコア / anomaly score（較正用 / calibration）")
        ax.set_ylabel("枚数 / images")
        ax.legend(fontsize=8)
        ax.set_title(f"threshold={threshold:.2f}   FP={table['FP']}  FN={table['FN']}", fontsize=9)
        plt.tight_layout()
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    slider = widgets.FloatSlider(value=0.50, min=0.00, max=1.00, step=0.01,
                                 description="threshold",
                                 continuous_update=False)
    widgets.interact(draw, threshold=slider)
    display(output)

### 実行結果の例 / Expected output

この結果は、この研修で保存した実行の記録です（モデル・データ分割・seed は本文に記載）。
細かな数値は環境で変わりますが、傾向と大小関係を読み取ってください。`x` のままの箇所は、
あなたの環境で決まる値です。

```text
分割の件数 / partition sizes: {'patchcore_train': 32, 'calibration': 26, 'test': 26}
分割の重なり / overlap check  : {'patchcore_train & calibration': 0, 'patchcore_train & test': 0, 'calibration & test': 0}
分割が決まらない行 / rows without a partition: 0 0

較正用 / calibration : 26 （正常 / normal: 6 , 異常 / abnormal: 20 ）
試験用 / test        : 26 （正常 / normal: 6 , 異常 / abnormal: 20 ）

較正用の正常品は 6 枚なので、誤報率 10% は「誤報 0 枚まで」という意味です。
目的 / objective: recall を最大に、ただし正常品の誤報率 ≤ 10%

method     threshold  recall  false_reject_rate  TP  FP  FN  TN
YOLO           0.xxx   x.xxx              x.xxx   x   x   x   x
PatchCore      0.xxx   x.xxx              x.xxx   x   x   x   x

試験用データでの結果（しきい値は固定済み）/ results on the untouched test partition
分母 / denominators: 異常 n= 20 , 正常 n= 6
```

These numbers come from the run recorded for this course (model, split and seed are stated in the
text). Exact values vary between machines; read the trends and the ordering. Where an `x` remains,
your own run decides the value.

The recorded run froze the two thresholds below. Read the first block as **calibration** (where the
thresholds were chosen) and the second as the **one test evaluation**:

```text
calibration (26 images, 6 normal / 20 abnormal)   threshold  recall  false_reject_rate  TP FP FN TN
YOLO                                                  0.060     1.00                0.00  20  0  0  6
PatchCore                                             0.500     1.00                0.00  20  0  0  6

test (26 images, 6 normal / 20 abnormal)          threshold  recall  false_reject_rate  TP FP FN TN
YOLO                                                  0.060     1.00                0.00  20  0  0  6
PatchCore                                             0.500     0.95                0.00  19  0  1  6
```


## Part 5　結果を並べて見る / Looking at them side by side

### 同じ画像を 5 つの列で並べる / five aligned columns for the same image

![](../../assets/diagrams/teach_score_map_overlay.png)

図：これは **並べ方の説明用** の 3 枚（元画像・異常マップ・重ね表示）です。**5 列の比較結果ではありません。**
実際の 5 列の比較は、次の図（この研修で保存した実行の記録）にあります。

Figure: this is the **layout explanation** only — three panels (original, score map, overlay). It is
**not** the five-column comparison. The real five-column result from the run recorded for this
course follows in the next figure.

**気づいてほしいこと**

比較は **同じ画像の同じ位置** を横に並べるのが原則です。列の意味を固定しないと、「どこが違うか」ではなく「どちらが良さそうか」という印象だけが残ります。並べる列は ①元画像 ②YOLO の四角 ③PatchCore の数値マップ ④正解マスク ⑤画像単位の判定 の 5 つです。**2 つの手法どうしが一致しているかだけを見ず、⑤の判定を④の正解と突き合わせてください。**

**What to notice**

Line up the **same image at the same position**. If the columns mean different things, you are left
with an impression of which looks better instead of knowing where they differ. The five columns are
① original, ② YOLO boxes, ③ PatchCore's numeric map, ④ the ground-truth mask, ⑤ the image-level
decision. **Do not compare the two methods only with each other: check the decisions in ⑤ against
the ground truth in ④.**

**考えてみてください / Try to answer**

- 5 つの列のうち、判定（⑤）に使ってはいけない列はどれですか。
- Which of the five columns must not be used for the decision (⑤)?


### 実際の 5 列の比較 / the real five-column comparison

![](../../assets/reference_outputs/ref_comparison_5col.png)

図：同じ 3 枚（試験用）を 5 列で並べた実際の結果。元画像・YOLO の四角・PatchCore の数値マップ・正解マスク・画像単位の判定。しきい値は固定済みで、この図のために動かしていません。

Figure: three test images in five aligned columns — original, YOLO boxes, PatchCore's numeric map, ground-truth mask and the image-level decision. The thresholds were already frozen and were not moved for this figure.

**気づいてほしいこと**

1 行目は 3 者が食い違っています（YOLO NG、PatchCore OK、正解 NG）。2・3 行目は一致です。列を増やす目的は「どちらが良いか」ではなく、**どこで食い違うか**を見ることです。比較の限界は 4 つあります：合成データと実データの差、少数（正常 6 枚・異常 20 枚）の不確かさ、学習・ラベルの予算の違い、解像度・前処理・しきい値への依存。

**What to notice**

Row 1 is a three-way disagreement (YOLO NG, PatchCore OK, truth NG); rows 2 and 3 agree. The columns exist to show **where the methods disagree**, not which is better. Four limits apply: the synthetic-to-real gap, the small sample (6 normal, 20 abnormal), different training and labelling budgets, and dependence on resolution, preprocessing and the operating threshold.

**考えてみてください / Try to answer**

- 1 行目の食い違いは、どちらの方法のどんな失敗ですか。
- In row 1, which method made which kind of mistake?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# ここに並ぶのは第5日ノートブック02 が保存した **較正用データ** の重ね表示です。
# 表示範囲は 0〜1 に固定して保存してあります / the saved overlays use the fixed 0-1 range.
# These overlays were saved from CALIBRATION images by Day 5 Notebook 02.
heatmap_dir = COURSE_ROOT / "submissions" / "day05" / "heatmaps"
heatmap_files = sorted(heatmap_dir.glob("*_heatmap.png")) if heatmap_dir.is_dir() else []

print("保存済みヒートマップ / saved heatmaps:", len(heatmap_files))
for path in heatmap_files:
    print("  ", path.name)

if heatmap_files:
    show([Image.open(path) for path in heatmap_files],
         [path.stem.replace("_heatmap", "") for path in heatmap_files],
         figsize=(4.2 * len(heatmap_files), 4.2))
else:
    print()
    print("ヒートマップがありません。第5日ノートブック02 を実行してください。")
    print("No heatmaps; run Day 5 Notebook 02.")


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 元の画像と並べる / compare with the original images
# 較正用の画像も test/ の下にあるので、同じ検索で元画像が見つかります。
# Calibration images also live under test/, so the same search finds the originals.
test_root = ASSETS / "anomaly_dataset" / "pvc_sheet" / "test"
if heatmap_files:
    pairs = []
    titles = []
    for path in heatmap_files[:4]:
        stem = path.stem.replace("_heatmap", "")
        original = next(test_root.rglob(f"{stem}.png"), None)
        if original is None:
            continue
        pairs.append(Image.open(original))
        titles.append(f"original: {stem}")
    if pairs:
        show(pairs, titles, figsize=(4.2 * len(pairs), 4.2))


#### 気づいてほしいこと
ヒートマップは **「どこが変か」** を示しますが、**「何が変か」** は示しません。

YOLO は **「何が」** を言えますが、**学習した種類しか** 言えません。

この 2 つは **競合しません**。組み合わせると、それぞれの弱点を補える場合があります。
ただし **「補完する」ことは保証されません**。組み合わせると計算時間と運用の手間が増え、
**新しい誤りの組み合わせ**（片方だけが誤る、2 つが別々に誤る）も生まれます。
組み合わせるかどうかは、**速度と誤りのバランスを測って決める設計判断** です。

この研修で保存した 5 列の図では、**一致しない行は 1 行だけ** です。それは PatchCore が
見逃した `streak_0005`（YOLO は NG、PatchCore は OK でスコア 0.464 < 0.50、正解は NG）で、
**PatchCore の見逃し 1 件** にあたります。ほかの行は 3 者が一致しています。

#### この比較の限界（4 つ）
この研修の比較から言えることには、次の 4 つの限界があります。

1. **合成画像と実画像の差（synthetic-to-real gap）** … 合成画像は欠陥がはっきりしており、
   背景のばらつきが小さい。実画像では同じ数字にならない。
2. **少ないサンプルによる不確かさ（small-sample uncertainty）** … 較正用・試験用とも正常 6 枚・
   異常 20 枚。1 枚の違いで率が大きく動く。
3. **学習とラベルの予算が違う（differing training/label budgets）** … YOLO は欠陥のラベル付き画像で
   学習し、PatchCore は正常画像だけで学習する。同じ条件の勝負ではない。
4. **解像度・前処理・運用しきい値への依存（resolution, preprocessing, operating threshold）** …
   入力サイズ、前処理、選んだしきい値が変われば結果も変わる。しきい値を動かせば見逃しと誤報の
   バランスも変わる。

だから、この比較は **「どちらが優れているか」の順位付けではなく、同じ画像でどこがどう違うかを
確かめるための材料** として読んでください。

#### What you should notice
A heatmap shows **where it looks unusual**, not **what is unusual**.

YOLO can say **what**, but **only for the classes it learned**.

The two do not compete; combining them can cover some of each other's weaknesses. **Complementarity
is not guaranteed, however.** Combining adds computation and operational work, and it creates
**new combinations of errors** (only one method wrong, or each wrong in a different case). Whether
to combine them is **a design decision to be made by measuring speed and errors**.

In the five-column figure recorded for this course, **exactly one row disagrees**: the image
PatchCore missed, `streak_0005` (YOLO NG, PatchCore OK with score 0.464 < 0.50, truth NG) — the
single PatchCore miss. Every other row agrees across all three.

#### The four limits of this comparison
Four limits bound what this comparison can tell you.

1. **Synthetic-to-real gap** — the synthetic images have crisp defects and little background
   variation; real images will not give the same numbers.
2. **Small-sample uncertainty** — 6 normal and 20 abnormal images in each partition; one image moves
   a rate a lot.
3. **Differing training and label budgets** — YOLO is fitted on labelled defect images, PatchCore on
   normal images only. This is not a like-for-like contest.
4. **Dependence on resolution, preprocessing and the operating threshold** — input size,
   preprocessing and the chosen threshold change the result, and moving the threshold trades misses
   against false alarms.

Read this comparison as **material for seeing where the two methods differ on the same images**,
not as a ranking of which method is better.


---

## 演習 / Exercises

#### 演習 1（基本）：比較表を完成させる
下の表を、あなたの結果で埋めてください。

| 項目 | YOLO | PatchCore |
|---|---|---|
| 学習に必要な画像 | | |
| 学習に必要なラベル | | |
| 出力 | | |
| 欠陥の種類が分かるか | | |
| 位置が分かるか | | |
| 見逃し（FN）の数 | | |
| 誤報（FP）の数 | | |

#### Exercise 1 (basic): complete the comparison table
Fill in the table with your own results.
Done when every cell is filled with a concrete answer or a number from your run.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
rows = [
    ("学習に必要な画像 / training images", "", ""),
    ("学習に必要なラベル / labels", "", ""),
    ("出力 / output", "", ""),
    ("欠陥の種類が分かるか / defect type", "", ""),
    ("位置が分かるか / location", "", ""),
    ("見逃し FN / misses", "", ""),
    ("誤報 FP / false alarms", "", ""),
]
for item, yolo_answer, patchcore_answer in rows:
    print(f"{item:<38} YOLO={yolo_answer or '-':<12} PatchCore={patchcore_answer or '-'}")

#### 演習 2（中級）：較正用データでしきい値を動かす

PatchCore のしきい値を 0.30 / 0.50 / 0.80 に変えて、**見逃し（FN）と誤報（FP）** が
どう変わるかを **3 行の表** にしてください。**しきい値の変更には較正用データだけを使います。**
試験用データで同じことをすると、その表は「答えを見てから採点した表」になります。

| しきい値 | 見逃し FN（分母: 異常 20） | 誤報 FP（分母: 正常 6） |
|---|---|---|
| 0.30 | | |
| 0.50 | | |
| 0.80 | | |

**できたら / Done when:** 3 行の表ができ、FN と FP が逆向きに動く（同時には動かないこともある）
ことを一文で書いた。件数が変わらない行があれば、その理由も書いてください。

#### Exercise 2 (intermediate): move the threshold on the calibration data

Vary the PatchCore threshold over 0.30 / 0.50 / 0.80 and record how the number of misses (FN) and
false alarms (FP) changes in a **three-row table**. **Change the threshold using calibration data
only.** Doing the same on the test partition would turn that table into marking your own exam.

**Done when:** the three-row table is complete and one sentence states that FN and FP move in
opposite directions (not necessarily at the same time). If a row's counts do not change, say why.


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# 演習 2 / Exercise 2
# 較正用データでだけ動かします。試験用データは汚しません。
# Explore on the CALIBRATION partition only; the test partition stays untouched.
thresholds_to_try = [0.30, 0.50, 0.80]      # ← PatchCore のしきい値 / anomaly thresholds

if len(calibration_frame):
    for threshold in thresholds_to_try:
        predicted = calibration_frame["score"] >= threshold
        truth = calibration_frame["truth_ng"]
        fn = int((~predicted & truth).sum())
        fp = int((predicted & ~truth).sum())
        print(f"calibration threshold={threshold:.2f}  FN={fn}  FP={fp}")
    print()
    print("試験用データの結果は変わりません。しきい値を決め直したら、新しい試験データが必要です。")
    print("The test results do not change. Re-tuning a threshold means you need fresh test data.")
else:
    print("較正用データがありません / no calibration data")

#### 演習 3（発展・任意）：PVC 検査への推奨を作る
次の構成で、**あなたの推奨** を書いてください。根拠として、
この 5 日間で得た数字や観察を必ず引用してください。

| 項目 | あなたの推奨 |
|---|---|
| 第 1 段階で使う手法 | |
| その理由 | |
| 第 2 段階で追加する手法 | |
| 見逃しを減らすためにすること | |
| 誤報を減らすためにすること | |
| 第 2 週で確認すべきこと | |

#### Exercise 3 (optional challenge): build a recommendation for PVC
Write your recommendation in the table above, citing numbers or observations from the
past five days as evidence.
Done when every row is filled and at least two rows quote a concrete result from this course.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
recommendation = {
    "stage1_method": "",
    "reason": "",
    "stage2_method": "",
    "reduce_misses": "",
    "reduce_false_alarms": "",
    "week2_checks": "",
}
for key, value in recommendation.items():
    print(f"{key:<22} {value or '(未記入 / not written)'}")

---

#### 理解度チェック / Understanding questions

1. YOLO の mAP と PatchCore の AUROC を直接比べてはいけないのはなぜですか。
2. 画像単位で比べるために、両方に何が必要ですか。
3. PatchCore が見逃しやすいのは、どんな欠陥ですか。
4. YOLO が見逃しやすいのは、どんな欠陥ですか。
5. あなたの現場なら、どちらを先に試しますか。理由も書いてください。

1. Why must YOLO's mAP and PatchCore's AUROC not be compared directly?
2. What does each method need in order to be compared at image level?
3. What kind of defect is PatchCore likely to miss?
4. What kind of defect is YOLO likely to miss?
5. On your line, which would you try first, and why?
Model answers: `docs/instructor/day05_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `yolo_predictions_on_shared_set.json` が無い | `yolo_predictions_on_shared_set.json` is missing | 第4日ノートブック03 の Part 5 を実行してください。 | Run Part 5 of Day 4 Notebook 03. |
| `patchcore_predictions.json` が無い | `patchcore_predictions.json` is missing | 第5日ノートブック02 の Part 5 を実行してください。 | Run Part 5 of Day 5 Notebook 02. |
| 共通の画像が 0 枚になる | Zero common images | パスの形式が違う可能性があります。`image` 列の先頭 `test/` を外しているか確認してください。 | Path formats may differ. Check that the leading `test/` was stripped. |
| precision や recall が `nan` になる | precision/recall are `nan` | TP + FP または TP + FN が 0 です。異常画像が 1 枚も無い、または 1 枚も検出していません。 | TP + FP or TP + FN is zero: no abnormal images, or nothing detected. |

## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] 2 つの結果ファイルを読み込めた
      - [ ] Loaded both result files
- [ ] 同じ画像で比較表を作った
      - [ ] Built the comparison table on shared images
- [ ] しきい値を動かして FN / FP の変化を見た
      - [ ] Moved thresholds and observed FN/FP
- [ ] 比較の限界を 4 つ説明できる
      - [ ] Can explain the four limits of the comparison
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [Ultralytics 検証モード](https://docs.ultralytics.com/modes/val/)
- [MVTec AD ベンチマーク](https://paperswithcode.com/sota/anomaly-detection-on-mvtec-ad)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

同じ画像に対して、YOLO で検出した位置と PatchCore のヒートマップの最大点がどれくらい離れているかを、3 枚で測ってみてください。

For three images, measure how far the YOLO box centre is from the brightest point of the PatchCore heatmap.